# Statistics for "Production Rate Calibration of in situ cosmogenic 14C in Antarctica"
The code below loads data from the supplemental materials to perform descriptive and inferential statistics on measurements of the CRONUS-A intercomparison material.

## Set-Up

### Imports

In [1]:
try: 
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    from scipy.stats import f_oneway, levene, bartlett, shapiro, t
    import pingouin as pg
except ImportError as e:
    missing_module = str(e).split()[-1]
    raise RuntimeError(f"Error: Missing module {missing_module}. Please install it to continue.")

### Classes and Functions

#### Classes

In [2]:
# Class Pub to store data for each lab or publication and perform the publication specific statistics
class Pub:
    def __init__(self, name, data):
        self.name = name
        if not isinstance(data, np.ndarray):
            try:
                self.data = np.array(data)
            except Exception as e:
                raise ValueError("Data must be convertible to a numpy array") from e
        else:
            self.data = data 
    def __repr__(self):
        return f"Pub: {self.name}"
    def append_stats(self):
        # Number of Measurements
        num_measurements = len(self.data)

        # Calculate Mean and Uncertainty
        if num_measurements > 1:
            mean_val = np.mean(self.data[:, 0]) # Calculate mean 
            prop_measure_unc = np.sqrt(np.sum(self.data[:, 1]**2)/num_measurements) # Propagated uncertainty of the mean (Eq. 3.13 Bevington and Robinson). Modified for comparison to standard Deviation of data by multiplying by sqrt(n)
            std_dv = np.std(self.data[:, 0], ddof=1) # Calculate standard deviation
            actual_unc = max(std_dv, prop_measure_unc) # Find max of uncertainties (Supplementary Materials Section 1.5)
            if prop_measure_unc > std_dv:
                print(f"Warning: For {self.name}, Propagated Measurement Uncertainty ({prop_measure_unc}) is greater than the Standard Deviation ({std_dv}).")
        else: # If the data is reported as summary statistics
            if self.name == 'Pub 3M':
                num_measurements = 6 # Set n for Cologne to six data points
            mean_val = self.data[:, 0][0] # Pull mean directly from data
            std_dv = self.data[:, 1][0] # Pull the Standard Deviation directly from data
            actual_unc = std_dv # Only option without raw measurement data
            prop_measure_unc = np.nan # No data to propagate if just summary statistics

        # Calculate Group Weighting Factor (Eq. 4.17 Bevington and Robinson)
        groupweight = 1/(actual_unc/np.sqrt(num_measurements))**2

        # Return results as array
        return np.asarray([mean_val, actual_unc, groupweight, num_measurements, std_dv, prop_measure_unc])

#### Functions

In [3]:
# Initialize reads the data from the supplementary spreadsheet (described in Supplementary Materials Section 1.3) and loads each publication into a class 
def initialize(filename, sheet=None, rows_to_skip=0, cols_to_use=None, flag_col='flag'):
    # Function to load data and separate by publication
    try:
        # Read full data into a pandas dataframe
        data = pd.read_excel(filename, skiprows=rows_to_skip, usecols=cols_to_use, sheet_name=sheet)

        # Handle issues with column names and missing data
        flag_col_actual = [c for c in data.columns if c.startswith('flag')][0]
        conc_col_actual = [c for c in data.columns if c.startswith('conc') and not c.startswith('conc_unc')][0]
        conc_unc_col_actual = [c for c in data.columns if c.startswith('conc_unc')][0]
        data = data.rename(columns={
            flag_col_actual: 'flag',
            conc_col_actual: 'conc',
            conc_unc_col_actual: 'conc_unc'
        })
        data = data.dropna(subset=['flag', 'conc', 'conc_unc'])
        classes = data['flag'].unique()
        pubs = {}

        # Create a Pub class for each publication and store in a dictionary
        for cls in classes:
            pubs[f'pub{cls}'] = Pub(name=f'Pub {cls}', data=data[data['flag'] == cls][['conc', 'conc_unc']].values)
        return pubs, data
    except Exception as e:
        raise ValueError("Error loading data from file") from e
    
# Calculate publication specific statistics and return a summary dataframe
def calculate_stats(pubs):
    df = []

    # Calculate statistics for each publication and append to the summary dataframe
    for pub_name, pub in pubs.items():
        stats = pub.append_stats()
        df.append({'Publication': pub_name, 'Mean': stats[0], 'Uncertainty': stats[1], 'Weight': stats[2], 'Relative Weight': np.nan, 'n': stats[3], 'Std Dev of Data': stats[4], 'Propagated Measurement Uncertainty': stats[5]})
    summary_data = pd.DataFrame(df)
    return summary_data

# Calculate the weighted mean and uncertainty of the full dataset
def calculate_weighted_mean(data):
    N = np.sum(data['n'].values) # Total number of measurements across all publications
    x = data['Mean'].values # Mean values from each publication
    w = data['Weight'].values # Weights from each publication
    sum_weights = np.sum(w) # Total weight across all publications
    weighted_mean = np.sum(w*x)/sum_weights # Calculate inverse-variance weighted mean (4.17 in Bevington and Robinson)
    std_err = np.sqrt(1/sum_weights) # Calculate the uncertianty of the mean (Eq. 4.19 in Bevington and Robinson)
    avg_std_samp = np.sqrt((((np.sum(w*(x**2))/sum_weights)-weighted_mean**2)*(N/(N-1)))) # Calculate the average deviation of the data (Eq. 4.22 in Bevington and Robinson)
    avg_std_mu = avg_std_samp/np.sqrt(N) # Calculate the average deviation of the mean (Eq. 4.23 in Bevington and Robinson)

    # Return the calculated values
    return weighted_mean, N, avg_std_samp, avg_std_mu, std_err, sum_weights

# Run the full analysis by combining all the functions above and returning the data
def run_full_analysis(filename, sheet=None, rows_to_skip=0, cols_to_use=None, flag_col='flag'):
    # Load data and initialize publications
    pubs, data = initialize(filename, sheet, rows_to_skip, cols_to_use, flag_col)

    # Calculate publication specific statistics and summary data
    summary_data = calculate_stats(pubs)

    # Calculate the weighted mean and uncertainties for the full dataset
    weighted_mean, N, samp_unc, mean_unc1, mean_unc2, sum_weights = calculate_weighted_mean(summary_data)

    # Calculate the relative weights for each publication and store in the summary dataframe
    summary_data['Relative Weight'] = summary_data['Weight']/sum_weights
    summary_stats = {'Weighted Mean': weighted_mean, 'Standard Error': mean_unc2, 'Average Deviation of Mean': mean_unc1,'Average Deviation of Data': samp_unc, 'N': N}
    

    # Print the summary statistics and return the results
    print(summary_data)
    print(f"Weighted Mean: {weighted_mean}, Standard Error: {mean_unc2}, Weighted Average Deviation of the Mean: {mean_unc1}, Weighted Average Deviation of the Data: {samp_unc}")

    return pubs, data, summary_stats, summary_data

## Data Processing

### CRONUS-A Compilation

In [4]:
pubs, data, summary_stats, summary_data = run_full_analysis('PR_Data.xlsx', sheet='Processing', cols_to_use='C:E')

  Publication           Mean   Uncertainty        Weight  Relative Weight  \
0        pub1  652263.333333  32734.260136  5.599461e-09         0.024865   
1        pub2  688516.666667  15420.711181  2.523146e-08         0.112043   
2       pub3M  672000.000000  71000.000000  1.190240e-09         0.005285   
3        pub4  709410.901438  38971.459147  8.559532e-09         0.038009   
4        pub5  727071.428571   7220.110802  1.342797e-07         0.596281   
5        pub6  668404.285714  66390.671318  3.176244e-09         0.014104   
6        pub7  705600.000000  27724.582954  1.170880e-08         0.051994   
7        pub8  601279.688979  50642.772118  6.628480e-09         0.029434   
8        pub9  708040.000000  16660.505909  2.882130e-08         0.127984   

      n  Std Dev of Data  Propagated Measurement Uncertainty  
0   6.0     32734.260136                        18351.315121  
1   6.0     10340.873593                        15420.711181  
2   6.0     71000.000000                

### Tulane dataset (Goehring et al., 2019)

In [5]:
goeh2019_pubs, goeh2019_data, goeh2019_summary_stats, goeh2019_summary_data = run_full_analysis('PR_Data.xlsx', sheet='Processing', cols_to_use='I:K')

  Publication      Mean   Uncertainty        Weight  Relative Weight     n  \
0      pub8.0  612600.0  31041.907158  1.037775e-08              1.0  10.0   

   Std Dev of Data  Propagated Measurement Uncertainty  
0     31041.907158                        21943.108257  
Weighted Mean: 612600.0, Standard Error: 9816.31295344642, Weighted Average Deviation of the Mean: 0.0, Weighted Average Deviation of the Data: 0.0


### Original Consensus Dataset (Jull et al., 2015)

In [6]:
int_pubs, int_data, int_summary_stats, int_summary_data = run_full_analysis('PR_Data.xlsx', sheet='Processing', cols_to_use='O:Q')

  Publication           Mean   Uncertainty        Weight  Relative Weight  \
0      pub1.0  725000.000000  35870.136140  5.440415e-09         0.056643   
1      pub2.0  651666.666667  32690.467520  5.614473e-09         0.058456   
2      pub3.0  693375.000000  40031.906917  4.992033e-09         0.051975   
3      pub4.0  690000.000000   5000.000000  8.000000e-08         0.832926   

     n  Std Dev of Data  Propagated Measurement Uncertainty  
0  7.0     35870.136140                        18083.141320  
1  6.0     32690.467520                        18434.568976  
2  8.0     40031.906917                         6763.874629  
3  2.0      2828.427125                         5000.000000  
Weighted Mean: 689917.1359033668, Standard Error: 3226.697690959476, Weighted Average Deviation of the Mean: 2661.769278111275, Weighted Average Deviation of the Data: 12765.397011751393


### CRONUS-N Compilation

In [7]:
CN_pubs, CN_data, CN_summary_stats, CN_summary_data = run_full_analysis('PR_Data.xlsx', sheet='Processing', cols_to_use='U:W')

  Publication         Mean   Uncertainty        Weight  Relative Weight    n  \
0      pub1.0  12727.80174   6990.584913  1.023159e-07         0.071411  5.0   
1      pub2.0  12375.00000   1744.276354  1.314708e-06         0.917598  4.0   
2      pub3.0  32576.00000  15937.553033  1.574768e-08         0.010991  4.0   

   Std Dev of Data  Propagated Measurement Uncertainty  
0      6990.584913                         3592.894500  
1      1744.276354                         1336.974196  
2     15937.553033                         3565.715005  
Weighted Mean: 12622.224420377332, Standard Error: 835.4327195669879, Weighted Average Deviation of the Mean: 607.7969534226296, Weighted Average Deviation of the Data: 2191.443080636089


## Inferential Statistics

### CRONUS-A Compilation

#### Normality and Homogeneity of Variance

In [8]:
# Test for normality using the Shapiro-Wilk test for both the compilation data and the Jull et al. (2015) data
conc_shapiro = shapiro(data['conc'])
int_shapiro = shapiro(int_data['conc'])

# Print the results of the Shapiro-Wilk test for both datasets
print(f'Shapiro-Wilk test for compilation. Statistic = {conc_shapiro.statistic}, p-value = {conc_shapiro.pvalue}')
print(f'Shapiro-Wilk test for Jull et al. (2015). Statistic = {int_shapiro.statistic}, p-value = {int_shapiro.pvalue}')

# Since the compilation data is not normal, then we have to use Levene to compare variances
stat, p = levene(data['conc'], int_data['conc'], center='mean')
if p < 0.05:
    print(f"\nThe variances between publications are not homogeneous. Suggest using Welch's t-test. (Statistic = {stat}, p-value={p})")
else:
    print(f"\nThe variances between publications are homogeneous. (Statistic = {stat}, p-value={p})")

Shapiro-Wilk test for compilation. Statistic = 0.9429275140754538, p-value = 0.0012795671499534408
Shapiro-Wilk test for Jull et al. (2015). Statistic = 0.9853863066926608, p-value = 0.9747424441605983

The variances between publications are not homogeneous. Suggest using Welch's t-test. (Statistic = 4.342004703131595, p-value=0.039682468541188294)


#### Independence Test

In [9]:
jull = pd.DataFrame({'Mean':int_summary_stats['Weighted Mean'], 'SD':int_summary_stats['Average Deviation of Data'], 'N':int_summary_stats['N'], 'SE':int_summary_stats['Standard Error']}, index=['Jull'])
n1 = summary_stats['N']
n2 = jull['N'].values[0]

# Comparison of our updated compilation mean to the Jull et al. (2015) consensus value using Welch's t-test
se_combined = np.sqrt((summary_stats['Average Deviation of Data']**2/n1) + (jull['SD'].values[0]**2/n2))
u = jull['SD'].values[0]**2/summary_stats['Average Deviation of Data']**2
t_stat = (summary_stats['Weighted Mean'] - jull['Mean'].values[0])/se_combined
df = (((1/n1)+(u/n2))**2)/((1/((n1**2)*(n1-1)))+((u**2)/((n2**2)*(n2-1))))
p_value = 2*(t.sf(np.abs(t_stat), df))

if p_value < 0.05:
    print(f'The difference between the weighted compilation mean and weighted mean of the Jull et al. (2015) data is significant. (t = {t_stat}, p = {p_value}).')
else:
    print(f'The difference between the weighted compilation mean and weighted mean of the Jull et al. (2015) data is not significant (t = {t_stat}, p = {p_value}).')

The difference between the weighted compilation mean and weighted mean of the Jull et al. (2015) data is significant. (t = 5.632200812436958, p = 2.912890823878152e-07).


### Publication Comparison

#### Welch's ANOVA 

In [10]:
# Interpublication statistical tests for unequal variances (Welch's ANOVA and post-hoc Games Howell HSD)

# Set random seed
rng = np.random.default_rng(0)

# Build groups for each publication and perform Shapiro-Wilk test for normality
groups = []
for flag, group in data.groupby('flag'):
    if len(group) > 1:
        groups.append(group['conc'].tolist())

        # Perform Shapiro-Wilk test for normality on the group
        shapiro_test = shapiro(group['conc'])
        if shapiro_test.pvalue < 0.05:
            print(f'Shapiro-Wilk test for Pub{flag} is NOT normal. Statistic = {shapiro_test.statistic}, p-value = {shapiro_test.pvalue}')
        else:
            print(f'Shapiro-Wilk test for Pub{flag} is normal. Statistic = {shapiro_test.statistic}, p-value = {shapiro_test.pvalue}')
    else:
        # This flag has only a mean reported (e.g. flag 3 / Cologne / Pub 3M).
        # Look up its precomputed mean, uncertainty, and n from summary_data,
        # and generate artificial data instead of using the single value in the table.
        # Don't calculate shapiro-wilk test becuase it will be normal.
        row = summary_data.loc[summary_data['Publication'] == f'pub{flag}'].iloc[0]
        mean, sd, n = row['Mean'], row['Uncertainty'], int(row['n'])
        x = rng.standard_normal(n)
        x = (x - x.mean()) / x.std(ddof=1)   # z-scores
        x = mean + sd * x                     # artificial data from mean and uncertainty
        groups.append(x.tolist())

# Perform Bartlett's test for homogeneity of variances across the groups
stat, p = bartlett(*groups)
if p < 0.05:
    print(f"\nThe variances between publications are not homogeneous. Use Welch's ANOVA. (Statistic = {stat}, p-value={p})")

    # Build long-format dataframe for pingouin from the same groups/flags
    long_rows = []
    flags_in_order = [flag for flag, group in data.groupby('flag')]
    for flag, vals in zip(flags_in_order, groups):
        for v in vals:
            long_rows.append({'conc': v, 'pub': f'pub{flag}'})
    df_long = pd.DataFrame(long_rows)

    # Perform Welch's ANOVA using pingouin
    anova_pingouin = pg.welch_anova(data=df_long, dv='conc', between='pub')
    print("\nWelch's ANOVA results (pingouin):")
    print(anova_pingouin)

    # Format and print the F-statistic, degrees of freedom, and p-value
    f_stat = anova_pingouin['F'].values[0]
    df1 = anova_pingouin['ddof1'].values[0]
    df2 = anova_pingouin['ddof2'].values[0]
    p_val = anova_pingouin['p_unc'].values[0]
    print(f"F({df1}, {df2:.2f}) = {f_stat:.2f}, p = {p_val:.3f}")
else:
    print(f"\nVariances are homogeneous. Standard ANOVA is appropriate. (Statistic = {stat}, p-value={p})")
    anova_results = f_oneway(*groups)

Shapiro-Wilk test for Pub1 is normal. Statistic = 0.8401810791453663, p-value = 0.13081583890247359
Shapiro-Wilk test for Pub2 is normal. Statistic = 0.9690938397148482, p-value = 0.8862929301443265
Shapiro-Wilk test for Pub4 is normal. Statistic = 0.9271086524326773, p-value = 0.3122408822758234
Shapiro-Wilk test for Pub5 is normal. Statistic = 0.937045915558792, p-value = 0.6122733242278262
Shapiro-Wilk test for Pub6 is normal. Statistic = 0.9203310635565614, p-value = 0.2223194209519693
Shapiro-Wilk test for Pub7 is normal. Statistic = 0.9026280552200231, p-value = 0.2676266342655541
Shapiro-Wilk test for Pub8 is normal. Statistic = 0.9530199523928515, p-value = 0.505996464532694
Shapiro-Wilk test for Pub9 is normal. Statistic = 0.9195268610820697, p-value = 0.4260360411321894

The variances between publications are not homogeneous. Use Welch's ANOVA. (Statistic = 55.89979254286483, p-value=2.9504722472203273e-09)

Welch's ANOVA results (pingouin):
  Source  ddof1      ddof2        

#### Post-hoc Games-Howell Test

In [11]:
gh_results = pg.pairwise_gameshowell(data=df_long, dv='conc', between='pub')
print(gh_results[['A', 'B', 'diff','pval']])

        A      B           diff          pval
0    pub1   pub2  -36253.333333  3.433053e-01
1    pub1  pub3M  -19736.666667  9.986794e-01
2    pub1   pub4  -57147.568105  9.692863e-02
3    pub1   pub5  -74808.095238  2.671754e-02
4    pub1   pub6  -16140.952381  9.975710e-01
5    pub1   pub7  -53336.666667  1.191677e-01
6    pub1   pub8   50983.644354  1.961321e-01
7    pub1   pub9  -55776.666667  8.139222e-02
8    pub2  pub3M   16516.666667  9.991000e-01
9    pub2   pub4  -20894.234771  6.816182e-01
10   pub2   pub5  -38554.761905  2.247080e-03
11   pub2   pub6   20112.380952  9.645184e-01
12   pub2   pub7  -17083.333333  7.463551e-01
13   pub2   pub8   87236.977687  5.608553e-05
14   pub2   pub9  -19523.333333  2.459680e-01
15  pub3M   pub4  -37410.901438  9.288801e-01
16  pub3M   pub5  -55071.428571  6.409545e-01
17  pub3M   pub6    3595.714286  1.000000e+00
18  pub3M   pub7  -33600.000000  9.529998e-01
19  pub3M   pub8   70720.311021  4.662746e-01
20  pub3M   pub9  -36040.000000  9

##### Print Significant Results for easy identification

In [12]:
sig_pairs = gh_results[gh_results['pval'] < 0.05]
print(sig_pairs[['A', 'B', 'diff', 'pval']])

       A     B           diff          pval
3   pub1  pub5  -74808.095238  2.671754e-02
10  pub2  pub5  -38554.761905  2.247080e-03
13  pub2  pub8   87236.977687  5.608553e-05
24  pub4  pub8  108131.212458  1.159459e-05
28  pub5  pub8  125791.739592  5.288467e-07
33  pub7  pub8  104320.311021  1.609999e-05
35  pub8  pub9 -106760.311021  2.923811e-06


### CRONUS-N

#### Welch's ANOVA

In [13]:
CN_flags_in_order = [flag for flag, group in CN_data.groupby('flag')]
CN_groups = [group['conc'].tolist() for _, group in CN_data.groupby('flag')]
for vals, flag in zip(CN_groups, CN_flags_in_order):
    shapiro_test = shapiro(vals)
    if shapiro_test.pvalue < 0.05:
        print(f'Shapiro-Wilk test for Pub{flag} is NOT normal. Statistic = {shapiro_test.statistic}, p-value = {shapiro_test.pvalue}')
    else:
        print(f'Shapiro-Wilk test for Pub{flag} is normal. Statistic = {shapiro_test.statistic}, p-value = {shapiro_test.pvalue}')

stat, p = bartlett(*CN_groups) # Because all of these are considered normal use Bartlett
if p < 0.05:
    print(f"\nThe variances between publications are not homogeneous. Use Welch's ANOVA. (Statistic = {stat}, p-value={p})")
    
    long_rows = []
    flags_in_order = [flag for flag, group in CN_data.groupby('flag')]
    for flag, vals in zip(flags_in_order, CN_groups):
        for v in vals:
            long_rows.append({'conc': v, 'pub': f'pub{flag}'})
    df_long = pd.DataFrame(long_rows)

    anova_pingouin = pg.welch_anova(data=df_long, dv='conc', between='pub')
    print("\nWelch's ANOVA results (pingouin):")
    print(anova_pingouin)

    f_stat = anova_pingouin['F'].values[0]
    df1 = anova_pingouin['ddof1'].values[0]
    df2 = anova_pingouin['ddof2'].values[0]
    p_val = anova_pingouin['p_unc'].values[0]
    print(f"F({df1}, {df2:.2f}) = {f_stat:.2f}, p = {p_val:.3f}")
else:
    print(f"\nVariances are homogeneous. Standard ANOVA is appropriate. (Statistic = {stat}, p-value={p})")
    anova_results = f_oneway(*CN_groups)

Shapiro-Wilk test for Pub1.0 is normal. Statistic = 0.9534005308291285, p-value = 0.7614450828112683
Shapiro-Wilk test for Pub2.0 is normal. Statistic = 0.8940979148077555, p-value = 0.4023378734641623
Shapiro-Wilk test for Pub3.0 is normal. Statistic = 0.9510925476453256, p-value = 0.7229462276418906

The variances between publications are not homogeneous. Use Welch's ANOVA. (Statistic = 8.982884844338178, p-value=0.01120447057132953)

Welch's ANOVA results (pingouin):
  Source  ddof1     ddof2         F     p_unc       np2
0    pub      2  4.903052  2.795623  0.154811  0.534192
F(2, 4.90) = 2.80, p = 0.155
